<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/02%20-%20Supervised%20Learning/02a_regression.ipynb)

# 02a · 监督学习入门：从分子特征预测水溶解度（回归）

在第一部分（`01a`–`01d`）结束时，我们已经能够把一个分子的化学结构（SMILES）转换成一排描述物理化学性质的数字（描述符）或一串 2048 维的二进制向量（分子指纹）。从这一章开始，我们正式进入**机器学习建模**：让计算机从成千上万个“分子特征 $\to$ 实验溶解度”的已知样本中自动归纳定量规律，再去预测未知分子的性质。

本章我们将全程使用 Python 传统机器学习的标准工具库 **[scikit-learn](https://scikit-learn.org/stable/user_guide.html)**（简称 `sklearn`）以及梯度提升树库 **XGBoost**，在第一部分清洗好的 **AqSolDB 水溶解度数据集**（9480 个有效分子）上完成全套训练、评估与诊断。

### 本节内容
1. 什么是监督学习（回归 vs 分类）与建模标准流程
2. 准备数据（特征矩阵 $X$ 与目标向量 $y$）与划分训练集/测试集
3. 回归评估指标（MAE、RMSE、$R^2$）与“瞎猜均值”基线
4. 第一个模型：多元线性回归与对角线图（Parity Plot）
5. 基于距离的模型：k 近邻（kNN）、特征缩放与数据泄漏防范
6. 决策树集成模型：随机森林、XGBoost 与过拟合诊断
7. 分子表示对比：物理化学描述符 vs 二值指纹 vs 计数指纹
8. 实战推理：预测并苯系列溶解度（初探树模型的外推盲区）

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "02 - Supervised Learning"))
    get_ipython().run_line_magic("pip", "install -q rdkit xgboost")


## 0. 环境检查与导入

Google Colab 已预装 `scikit-learn` 和 `xgboost`，只需按需安装 `rdkit`：

In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit
try:
    import xgboost
except ImportError:
    %pip install -q xgboost
    import xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")   # 关闭 RDKit 警告刷屏
print(f"RDKit: {rdkit.__version__} | scikit-learn: {sklearn.__version__} | XGBoost: {xgboost.__version__}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 1. 什么是监督学习？

**监督学习（Supervised Learning）** 是目前在材料与化学研发中落地最广泛的机器学习范式。它的任务很明确：给计算机喂入大量带有“标准答案（标签 $y$）”的样本特征 $X$，让它拟合出一个映射函数 $f(X) \approx y$，以便未来给定新材料的特征 $X_{\text{new}}$ 时能直接预测其性质。

根据“标准答案 $y$”的数据类型，监督学习分为两大类：

| 任务类型 | 预测目标 $y$ | 材料与化学中的典型问题 |
|---|---|---|
| **回归（Regression）** | **连续数值** | 水溶解度 $\log S$、HOMO/LUMO 能级、载流子迁移率、形成能、带隙 |
| **分类（Classification）** | **离散类别** | p 型还是 n 型半导体？能否溶液加工？薄膜退火后是否结晶？是否有毒？ |

这一节（`02a`）和下一节（`02b`）我们专注于**回归任务**（直接预测 $\log S$ 的具体数值），第 `02c` 节再讨论分类任务。

一个规范的监督学习项目，无论算法多简单或多复杂，都遵循以下六步流程：
1. **定义问题**：明确输入特征 $X$ 是什么、预测目标 $y$ 是什么；
2. **数据准备与清洗**：剔除无效结构，构建数值矩阵；
3. **划分数据集**：将已知数据切分为**训练集（Train）**和**测试集（Test）**；
4. **建立基线与评估指标**：明确用什么尺子衡量误差，以及“不学任何规律瞎猜”能得多少分；
5. **训练与对比不同模型**：从简单线性模型逐步过渡到非线性树集成模型；
6. **模型推理与物理合理性检验**：在新分子上测试，检查预测趋势是否符合化学常识。

下面我们就严格按照这个流程一步步走通。

---
## 2. 准备数据与划分训练集/测试集

### 2.1 读取并清洗 AqSolDB 数据

我们先读入 `data/solubility.csv`，并像 `01d` 第 5 节那样，用 `Chem.MolFromSmiles` 过滤掉那 2 条无法解析成合法分子图的脏数据：

In [ ]:
import os

DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请确保它在 data/ 目录下")

df = pd.read_csv(DATA_FILE)
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notna()].reset_index(drop=True)
print(f"清洗后共有 {len(df)} 个有效分子。")

### 2.2 构建特征矩阵 $X$ 与目标向量 $y$

在机器学习文献和 `scikit-learn` 代码里，大家约定俗成使用以下记号：
- **$X$（大写）**：**特征矩阵（Feature Matrix）**，形状为 `(样本数, 特征数)` 的二维数组。每一行代表一个分子，每一列代表一个描述符。
- **$y$（小写）**：**目标向量（Target Vector）**，形状为 `(样本数,)` 的一维数组，存放每个分子对应的实验 $\log S$。

> 注意：表里的 `Name` 和 `SMILES` 是文本字符串，不能直接放进数值矩阵 $X$ 里；我们取表中现成算好的 17 列 RDKit 描述符作为 $X$：

In [ ]:
feature_cols = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
    "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
    "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT"
]

X = df[feature_cols].values      # .values 将 DataFrame 转换为二维 NumPy 数组
y = df["Solubility"].values      # 一维 NumPy 数组

print("特征矩阵 X 的形状 (分子数, 描述符数):", X.shape)
print("目标向量 y 的形状 (分子数,):        ", y.shape)

### 2.3 划分训练集与测试集

评估一个机器学习模型的核心原则是：**绝不能拿训练时见过的数据来给模型打分**。就像期末考试如果原封不动考平时的课后作业，学生拿了满分也说明不了他真正掌握了知识（他可能只是把答案背下来了）。

因此在训练开始前，我们先用 `train_test_split` 随机抽出一部分数据（通常是 20%）锁进抽屉，作为**测试集（Test Set）**；剩下的 80% 作为**训练集（Training Set）**供模型学习：

| 数据子集 | 比例 | 用途 | 类比 |
|---|---|---|---|
| **训练集（Train）** | ~80% | 供模型拟合参数、总结规律 | 平时做的课本例题与练习册 |
| **验证集（Validation）** | （见 `02b`） | 挑选模型种类、调整超参数 | 考前模拟卷 |
| **测试集（Test）** | ~20% | 训练全部结束后，考评模型的泛化能力 | 期末正式考试（绝不用于调参） |

In [ ]:
from sklearn.model_selection import train_test_split

# test_size=0.2 表示留出 20% 作为测试集；random_state=42 固定随机打乱的种子，保证每次运行划分结果一致
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("训练集大小:", X_train.shape, "| 测试集大小:", X_test.shape)

---
## 3. 回归评估指标与“瞎猜均值”基线

模型对测试集里的 $n$ 个分子给出预测值 $\hat{y}_i$ 后，怎么跟真实实验值 $y_i$ 对比打分？回归任务最常用的三个指标如下：

| 指标 | 数学定义 | 物理含义与判读标准 |
|---|---|---|
| **MAE**（平均绝对误差） | $\frac{1}{n}\sum_{i=1}^n \lvert y_i - \hat{y}_i \rvert$ | 预测值与真实值平均相差多少个 $\log S$ 单位（**越小越好**） |
| **RMSE**（均方根误差） | $\sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2}$ | 先平方再平均最后开根号，与 $y$ 量纲相同，但**对个别预测错得离谱的样本惩罚更重**（**越小越好**） |
| **$R^2$**（决定系数） | $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | 模型解释了目标变量百分之多少的方差。**完美预测为 $1$**；如果和“永远猜平均值 $\bar{y}$”一样差，则为 **$0$** |

> **对误差量级的化学直觉**：$\log S$ 是以 10 为底的对数。如果模型的 $\text{RMSE} = 1.0$，意味着它预测的摩尔溶解度平均有 **$10^1 = 10$ 倍**（一个数量级）的偏差。

在训练任何真正的算法之前，我们先测试一个最笨的**零知识基线（Dummy Baseline）**：不管来什么分子，一律盲猜训练集溶解度的平均值 `y_train.mean()`。看看这个“瞎猜模型”能考多少分：

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_baseline = np.full_like(y_test, fill_value=y_train.mean())

print(f"瞎猜均值基线 | MAE  = {mean_absolute_error(y_test, y_baseline):.2f}")
print(f"瞎猜均值基线 | RMSE = {np.sqrt(mean_squared_error(y_test, y_baseline)):.2f}")
print(f"瞎猜均值基线 | R²   = {r2_score(y_test, y_baseline):.2f}")

记住 **`RMSE = 2.42`** 这个及格线：后面任何机器学习模型如果连 2.42 都跑不赢，说明它根本什么都没学到。

为了避免后面每换一个模型都要重复写五六行算指标的代码，我们用 `01a` 学的 `def` 封装一个通用的评估函数 `evaluate()`，并准备一个空字典 `results` 用来收集各个模型的测试集 RMSE：

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test, name="model"):
    """拟合模型，打印训练集与测试集的 RMSE 和测试集 R²，并返回测试集 RMSE。"""
    model.fit(X_train, y_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    y_pred = model.predict(X_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
    r2_test = r2_score(y_test, y_pred)
    print(f"{name:20s} | 训练 RMSE = {rmse_train:.2f} | 测试 RMSE = {rmse_test:.2f} | 测试 R² = {r2_test:.2f}")
    return rmse_test

results = {}

---
## 4. 第一个模型：多元线性回归与 scikit-learn 三步曲

### 4.1 scikit-learn 的统一接口

`scikit-learn` 最优秀的设计在于：**不管底层是简单的线性回归，还是复杂的随机森林、支持向量机，调用代码永远是完全相同的三步**：

```python
model = SomeModel(超参数)          # 1. 实例化一个模型对象
model.fit(X_train, y_train)        # 2. 在训练集上拟合（学习规律）
y_pred = model.predict(X_test)     # 3. 对新数据给出预测
```

我们先从最简单的**多元线性回归（`LinearRegression`）**开始。它假设水溶解度 $\log S$ 是 17 个描述符的线性加权和：
$$\log S \approx w_1 \cdot \text{MolWt} + w_2 \cdot \text{MolLogP} + \dots + w_{17} \cdot \text{BertzCT} + b$$

调用 `fit()` 时，模型通过最小二乘法解出使训练集平方误差最小的权重 $w_1, \dots, w_{17}$ 和截距 $b$：

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
results["Linear"] = evaluate(lin_reg, X_train, y_train, X_test, y_test, "Linear regression")

测试集 RMSE 从基线的 `2.42` 降到了 `1.64`，$R^2$ 达到 `0.54`——仅仅把 17 个描述符做线性加权，就已经能解释一半以上的溶解度变化了。

### 4.2 用对角线图（Parity Plot）直观检查预测质量

单看一个 `RMSE = 1.64`，我们并不知道模型是在所有分子上都均匀地差一点，还是在某些特定区间错得特别离谱。回归任务最标准的诊断图是**对角线散点图（Parity Plot）**：横轴画实验真实值 $y_{\text{true}}$，纵轴画模型预测值 $y_{\text{pred}}$。如果预测完美，所有点都应该严丝合缝地落在 $y = x$ 对角虚线上：

In [ ]:
def parity_plot(y_true, y_pred, title):
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.scatter(y_true, y_pred, s=8, alpha=0.35)
    lims = [min(y_true.min(), y_pred.min()) - 0.5, max(y_true.max(), y_pred.max()) + 0.5]
    ax.plot(lims, lims, "k--", lw=1, label="Ideal (y = x)")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("Measured logS (log mol/L)")
    ax.set_ylabel("Predicted logS (log mol/L)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

parity_plot(y_test, lin_reg.predict(X_test), "Linear regression (test set)")

从图中可以清楚看到：在中间区域（$-5 < \log S < 0$）点群大体沿着对角线分布，但在两端（极难溶的左下角和极易溶的右上角）明显偏离，甚至有个别分子被预测到了 $\log S < -20$。这说明分子结构与溶解度之间存在显著的**非线性关系**，一条刚性的直线力不从心。

### 4.3 检查线性模型的权重：兑现 `01c` 的“多重共线性”伏笔

线性回归训练完后，学到的 17 个权重保存在属性 `lin_reg.coef_` 里。我们把它们打印出来看看：

In [ ]:
coef = pd.Series(lin_reg.coef_, index=feature_cols).sort_values()
coef.round(3)

看上面这串数字时，千万不要直接拿权重的绝对值大小或正负号去写论文下物理结论！这里藏着两个经典的陷阱：
1. **特征量纲与数值范围不同**：比如 `MolWt` 的权重看起来只有微不足道的 `-0.004`，但这只是因为分子量动辄几百，乘上几百之后它对 $\log S$ 的实际贡献依然有 $-1 \sim -2$ 个单位。
2. **多重共线性导致权重符号失真**：还记得 `01c` 热图里发现的高度相关特征吗？看上面的 `HeavyAtomCount`（重原子数，权重 **`-0.501`**）和 `NumValenceElectrons`（价电子数，权重 **`+0.073`**）——它们俩描述的明明是同一件事（分子的体积大小），一个重原子大约对应 4~6 个价电子。正因为这两列高度共线，线性回归在凑最小二乘解时，给重原子数赋予了一个很大的负权重，同时又给价电子数赋予了一个正权重来互相抵消！**当输入特征存在强共线性时，线性回归的单个权重不再具有独立的物理可解释性。**

---
## 5. 基于距离的模型：k 近邻（kNN）、特征缩放与数据泄漏

既然一条全局直线不够灵活，我们换一种完全不假设全局公式的算法——**k 近邻回归（k-Nearest Neighbors, `KNeighborsRegressor`）**。

它的原理非常符合化学家的日常直觉（“相似分子具有相似性质”）：当来了一个新分子时，去训练集里找出跟它在 17 维特征空间里**几何距离最近的 $k$ 个分子**，直接拿这 $k$ 个邻居的实验 $\log S$ 的平均值作为预测结果。

### 🏋️ 练习 02a-1

从 `sklearn.neighbors` 中导入 `KNeighborsRegressor`，创建一个参考最近 **10** 个邻居的模型对象 `knn_reg = KNeighborsRegressor(n_neighbors=10)`，并调用上面的 `evaluate()` 函数看看它在原始特征 `X_train, X_test` 上的表现：

In [ ]:
# 在这里写你的代码


### 5.1 为什么基于距离的模型必须做特征缩放？

我们用默认的 $k=5$ 在原始特征上跑一下 kNN，并看一眼这 17 列特征各自的最小值和最大值：

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

results["kNN (raw)"] = evaluate(
    KNeighborsRegressor(n_neighbors=5), X_train, y_train, X_test, y_test, "kNN (raw features)"
)

df[feature_cols].describe().loc[["min", "max"]].round(1)

问题立刻暴露出来了：kNN 算两个分子的欧氏距离 $\sqrt{\sum_j (x_{1j} - x_{2j})^2}$ 时，是把每一列特征的差值平方直接加起来的。而在我们的表里，复杂度指标 `BertzCT` 的数值高达几千，分子量 `MolWt` 是几百，而最重要的疏水常数 `MolLogP` 和芳香环数 `NumAromaticRings` 只有个位数！

如果不做处理直接算距离，**几千的 `BertzCT` 差一点点，就会把 `MolLogP` 的差异彻底淹没**——等于模型只按 `BertzCT` 和 `MolWt` 找邻居，把数值小的关键特征完全忽略了。

解决办法是**标准化缩放（`StandardScaler`）**：把每一列特征都减去它自己的均值 $\mu$，再除以它自己的标准差 $\sigma$：
$$x' = \frac{x - \mu}{\sigma}$$
这样所有 17 列特征都变成了“均值为 0、标准差为 1”的无量纲数，站在同一条起跑线上参与距离计算：

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)                        # 关键：只用训练集计算均值和标准差！
X_train_s = scaler.transform(X_train)      # 用训练集的尺子缩放训练集
X_test_s = scaler.transform(X_test)        # 用同一把尺子缩放测试集

print("缩放后训练集前 5 列的均值:", X_train_s.mean(axis=0).round(2)[:5])
print("缩放后训练集前 5 列的标准差:", X_train_s.std(axis=0).round(2)[:5])

results["kNN (scaled)"] = evaluate(
    KNeighborsRegressor(n_neighbors=5), X_train_s, y_train, X_test_s, y_test, "kNN (scaled)"
)

仅仅加了一步标准化缩放，完全相同的 kNN 模型，测试集 RMSE 就从 `1.56` 大幅降到了 **`1.24`**，$R^2$ 从 `0.58` 跃升到 **`0.74`**！

> **防坑铁律：为什么 `scaler.fit()` 只能传 `X_train`，不能先缩放整个 `X` 再划分数据集？**
> 如果在 `train_test_split` 之前对全部数据做 `StandardScaler().fit_transform(X)`，那么计算均值 $\mu$ 和标准差 $\sigma$ 时就已经把测试集分子的分布信息偷偷混进去了。这种在训练阶段不小心用到了测试集信息的错误，统称为**数据泄漏（Data Leakage）**。
>
> 记住一条简单的准则：**测试集在训练完成前是不存在的。任何带有 `fit` 的数据预处理步骤（缩放、缺失值均值插补、PCA 降维、特征选择），都只能在 `X_train` 上 `fit`，然后对 `X_train` 和 `X_test` 分别调用 `transform`。**

### 🏋️ 练习 02a-2

支持向量回归（**SVR**, `from sklearn.svm import SVR`）同样是一种依赖特征空间距离/核函数的非线性模型，因此也必须使用缩放后的数据。请用默认参数实例化一个 `SVR()`，在缩放后的数据 `X_train_s, y_train, X_test_s, y_test` 上调用 `evaluate()`，并将返回的测试 RMSE 存入 `results["SVR"]`。

In [ ]:
# 在这里写你的代码


---
## 6. 决策树集成模型：随机森林、XGBoost 与过拟合诊断

在表格型科学数据（描述符表格）上，目前综合表现最强的传统机器学习算法通常不是距离模型，而是基于**决策树（Decision Tree）**的集成模型。

一棵决策树就像一连串层层递进的化学是非题：“第 1 步看 `MolLogP > 2.5` 吗？如果是，第 2 步再看 `TPSA < 40` 吗？……”一路分叉到最底部的叶子节点，用落入该叶子的训练样本平均值作为预测。因为决策树只看单列特征的排序切分点，所以它**完全不受特征量纲影响，不需要做特征缩放**。

单棵决策树很容易把训练数据死记硬背下来，因此实际应用中总是把成百上千棵树组合起来（称为**集成学习 Ensemble Learning**），主流有两种流派：
- **随机森林（Random Forest, `RandomForestRegressor`）**：同时并行训练上百棵深树（每棵树只随机抽一部分样本和一部分特征来看），最后让所有树**投票取平均**，靠集体智慧降低方差；
- **梯度提升树（Gradient Boosting, 代表库 `XGBRegressor`）**：串行地一棵接一棵训练浅树，**每一棵新树都专门去拟合前面所有树留下的预测残差（错误）**，积小胜为大胜。

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# n_estimators=100 表示训练 100 棵树；n_jobs=-1 表示调用所有 CPU 核心并行加速
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
results["Random forest"] = evaluate(rf, X_train, y_train, X_test, y_test, "Random forest")

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
results["XGBoost"] = evaluate(xgb, X_train, y_train, X_test, y_test, "XGBoost")

### 诊断过拟合（Overfitting）

仔细对比上面随机森林的两个数字：
- 在见过的**训练集**上，$\text{RMSE} = 0.42$（准得惊人）；
- 但在没见过的**测试集**上，$\text{RMSE} = 1.17$（误差放大了将近 3 倍）。

这种“**训练集上表现极好，但一到测试集上误差显著增大**”的现象，叫做**过拟合（Overfitting）**——因为默认不限深度的决策树会一直分叉，直到每个叶子里只剩一两个分子，相当于把训练集里的实验噪声和个别特殊分子都死记硬背了下来。

| 诊断现象 | 术语 | 常见原因与对策 |
|---|---|---|
| 训练误差大，测试误差也大 | **欠拟合（Underfitting）** | 模型太简单（如刚才的线性回归），抓不住复杂规律 $\to$ 换非线性模型或增加有效特征 |
| 训练误差极小，测试误差大得多 | **过拟合（Overfitting）** | 模型自由度太高、树太深、或训练样本太少 $\to$ 限制树深/正则化，或增加训练数据 |
| 训练误差小，测试误差也小且接近 | **拟合良好** | 理想状态 |

> 尽管随机森林存在一定程度的过拟合，但看最终标尺——**测试集 RMSE（`1.17`）**，它和 XGBoost 依然显著优于线性回归（`1.64`）和 kNN（`1.24`）。下一节 `02b` 我们会尝试通过限制树的最大深度来抑制过拟合。

我们把目前所有模型在测试集上的 RMSE 画成柱状图对比一下：

In [ ]:
def plot_results(results_dict):
    names = list(results_dict.keys())
    vals = [results_dict[n] for n in names]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(names, vals, color="steelblue")
    ax.axvline(2.42, color="gray", ls="--", label="Baseline (predict mean = 2.42)")
    ax.set_xlabel("Test RMSE (logS units, lower is better)")
    ax.invert_yaxis()
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_results(results)

---
## 7. 分子表示对比：物理化学描述符 vs 分子指纹

在 `01d` 讲 Morgan 指纹时我们留下了一个悬念：**把分子编码成什么样的特征，决定了模型能学到什么**。
- 前面我们一直用的是 **17 个 RDKit 物理化学描述符**（直接给出了分子量、`MolLogP`、`TPSA`、芳香环数等全局宏观性质）；
- 如果不用这 17 个描述符，改用 `01d` 学的 **2048 维 Morgan 分子指纹**（只告诉模型分子里有哪些半径 $r \le 2$ 的局部原子环境），随机森林能把溶解度预测得多准？
- 而且我们在 `01d` 发现，**二值指纹（Bit FP，只记 0/1）**分不清并四苯和并五苯，而**计数指纹（Count FP，记录每个局部片段出现的次数）**能分清。在近万个分子的回归任务上，计数指纹是不是真的比二值指纹更好？

我们马上动手做实验验证（注意：生成 9480 个分子的指纹并训练两个 2048 维的随机森林大约需要 30~60 秒）：

In [ ]:
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# 生成计数指纹矩阵 (9480, 2048)，大于 0 的位置转成 0/1 即得到二值指纹矩阵
fp_count = np.array([fp_gen.GetCountFingerprintAsNumPy(m) for m in df["mol"]])
fp_bit = (fp_count > 0).astype(np.uint8)
print("指纹特征矩阵形状:", fp_count.shape)

# 使用与前面 X 完全相同的 random_state=42 划分，确保训练集和测试集对应同一批分子
fpb_train, fpb_test = train_test_split(fp_bit, test_size=0.2, random_state=42)
fpc_train, fpc_test = train_test_split(fp_count, test_size=0.2, random_state=42)

In [ ]:
results["RF + bit FP"] = evaluate(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    fpb_train, y_train, fpb_test, y_test, "RF (bit FP)"
)

results["RF + count FP"] = evaluate(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    fpc_train, y_train, fpc_test, y_test, "RF (count FP)"
)

实验结果非常耐人寻味：
1. **计数指纹（`RMSE = 1.23`）确实显著优于二值指纹（`RMSE = 1.36`）**：对于溶解度这种强烈依赖分子尺寸和重复单元个数的性质，知道“有**几个**苯环、**几个**亚甲基 `CH2`”比单纯知道“有没有”重要得多。
2. **但单用 2048 维指纹（`1.23`），仍然不如单用短短 17 个物理化学描述符（`1.17`）**！为什么？因为水溶解度主要由分子的**全局疏水性（`MolLogP`）和极性表面积（`TPSA`）**决定；描述符直接把人类化学家总结了上百年的全局物理量喂给了模型，而如果只给局部碎片指纹，树模型得自己从两千多个碎片里去艰难地拼凑全局性质。

这自然引出了一个想法：如果把**宏观描述符**和**局部计数指纹**拼在一起呢？

### 🏋️ 练习 02a-3

用 `np.hstack([X_train, fpc_train])` 把 17 个描述符和 2048 维计数指纹横向拼接成 `2065` 维的组合特征（对测试集 `X_test` 和 `fpc_test` 做同样拼接），训练一个 `RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)`，并将测试集 RMSE 存入 `results["RF + desc + count FP"]`。看看“全局描述符 + 局部碎片指纹”强强联合后能不能刷新最佳纪录？

In [ ]:
# 在这里写你的代码


---
## 8. 实战推理：用训练好的模型预测并苯系列

模型训练好之后，我们最终要用它来预测感兴趣的新分子。注意：一个新分子拿过来时只有一个 SMILES 字符串，我们必须先用 RDKit 算出**与训练矩阵 `X_train` 顺序完全一致的 17 个描述符**，再交给模型 `predict`：

In [ ]:
def featurize(smiles):
    """将单个 SMILES 转换为形状为 (1, 17)、列顺序与 feature_cols 一致的描述符数组。"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"无法解析 SMILES: {smiles}")
    return np.array([[getattr(Descriptors, name)(mol) for name in feature_cols]])

# 用 17 个描述符训练好的随机森林预测苯的水溶解度：
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
print(f"苯 (benzene) 的预测 logS = {rf.predict(featurize('c1ccccc1'))[0]:.2f}")

现在，我们让模型一口气预测整个**并苯（acene）系列**——从单环的苯（`N=1`）一直到五环的并五苯（`N=5`），并把预测值与数据库里的实验测量值画在同一张图上对比（回想 `01b` 第 7 节，苯到并四苯都在 AqSolDB 里，唯独并五苯不在库里）：

In [ ]:
acenes = {
    "benzene": "c1ccccc1",
    "naphthalene": "c1ccc2ccccc2c1",
    "anthracene": "c1ccc2cc3ccccc3cc2c1",
    "tetracene": "c1ccc2cc3cc4ccccc4cc3cc2c1",
    "pentacene": "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
}

# 用规范 SMILES 查找该分子是否已存在于 AqSolDB 中
canon_to_logS = dict(zip(df["mol"].apply(Chem.MolToSmiles), df["Solubility"]))

rows = []
for n_rings, (name, smi) in enumerate(acenes.items(), start=1):
    canon_s = Chem.MolToSmiles(Chem.MolFromSmiles(smi))
    measured = canon_to_logS.get(canon_s)   # 库里没有则返回 None
    pred = rf.predict(featurize(smi))[0]
    rows.append({
        "molecule": name,
        "rings": n_rings,
        "measured logS": measured,
        "predicted logS": pred,
    })

acene_table = pd.DataFrame(rows).round(2)
acene_table

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.plot(acene_table["rings"], acene_table["measured logS"], "o-", label="Measured (in AqSolDB)")
ax.plot(acene_table["rings"], acene_table["predicted logS"], "s--", label="Random forest prediction")
ax.set_xlabel("Number of fused rings (N)")
ax.set_ylabel("logS (log mol/L)")
ax.set_xticks(range(1, 6))
ax.legend()
plt.tight_layout()
plt.show()

### 为什么随机森林在并五苯上“拐平”了？

观察上面的图和表：
- 从苯（`N=1`）到并四苯（`N=4`），每增加一个苯环，实验 $\log S$ 都很规律地下降约 $2$ 个单位。前四个分子因为本来就在数据库里，随机森林预测得相当准。
- 但到了真正没见过的**并五苯（`N=5`）**，按照物理化学趋势它的 $\log S$ 应该继续降到 $-10$ 左右；可随机森林给出的预测值（约 `-7.95`）却**几乎跟并四苯一模一样，完全不敢继续往下走**！

这不是偶然的 bug，而是所有**决策树集成模型（随机森林、梯度提升树）的固有数学特性**：
树的每一个叶子节点输出的都是落入该叶子的**已知训练样本的平均值**。因此，随机森林的预测值永远被锁死在训练数据的最小与最大值之间，**天然无法向训练数据范围之外做趋势外推（Extrapolation）**。在训练集里像并五苯这样的大稠环分子极其稀少，模型只能拿它能找到的最接近的邻居（如并四苯、苝）的溶解度来充数。

这给我们敲响了警钟：**测试集上的平均 RMSE 好看，绝不代表模型在你关心的每一类新分子上都可靠。** 下一节（`02b`）我们就专门来解决模型评估的可靠性、超参数调优与适用域边界问题。

---
## 小结

这一节我们跑通了回归建模的完整主线：
- **标准流程与接口**：构建数值矩阵 $X$ 和标签 $y$，用 `train_test_split` 留出测试集，先算“瞎猜均值”基线（$\text{RMSE} = 2.42$），再用 `fit` / `predict` 训练和评估模型；
- **三类经典回归算法**：
  1. **多元线性回归（$\text{RMSE} \approx 1.64$）**：简单快速，但抓不住非线性关系，且特征共线性会导致权重符号失真；
  2. **k 近邻与 SVR（缩放后 $\text{RMSE} \approx 1.24 \sim 1.29$）**：基于特征空间距离，**必须使用仅在训练集上 `fit` 的 `StandardScaler` 缩放特征**，严防数据泄漏；
  3. **随机森林与 XGBoost（$\text{RMSE} \approx 1.17$）**：无需特征缩放，在表格特征上精度最高，但需留意训练误差远小于测试误差的**过拟合**现象，以及无法在训练范围外做趋势**外推**的局限；
- **分子特征表示**：全局描述符（`1.17`）与局部计数指纹（`1.23`）各有千秋，二者拼接互补后取得了最佳效果（`1.11`）。

**下一节（`02b_validation_and_tuning.ipynb`）**：深究交叉验证、超参数网格搜索、Pipeline 防泄漏，以及怎样定量判断一个新分子有没有超出模型的“能力圈（适用域）”。